# Analysis of the Strogatz problems

## Prelude

In [ ]:
import pandas as pd
import seaborn as sns
import sympy

## Load data

In [ ]:
report = pd.read_csv("Generated/full-report.csv")

In [ ]:
report

In [ ]:
report["sympy_expr"] = report["expr_original_syms"].apply(lambda e: sympy.sympify(e))

## Definitions

In [ ]:
def get_data_set(data_set):
    return report[report["data_set"] == data_set]

In [ ]:
def parse_if_needed(expr_or_str) -> sympy.Expr:
    if isinstance(expr_or_str, str):
        return sympy.sympify(expr_or_str)
    elif isinstance(expr_or_str, sympy.Expr):
        return expr_or_str
    else:
        raise ValueError("Input must be a string or a sympy expression")

In [ ]:
def replace_near_integer(expr, tolerance=1e-5):
    if expr.func == sympy.Float:
        x = expr.evalf()
        x_int = round(x)
        x_frac = x - x_int
        if abs(x_frac) < tolerance:
            return sympy.Integer(x_int)
        else:
            return expr
    elif len(expr.args) == 0:
        return expr
    else:
        new_args = map(
            lambda e: replace_near_integer(e, tolerance=tolerance), expr.args
        )
        return expr.func(*new_args)

In [ ]:
def to_spiffy(expr):
    expr = sympy.expand(expr, rational=False)
    expr = replace_near_integer(expr, tolerance=1e-6)
    # Too time consuming:
    # expr = sympy.simplify(expr)
    return expr

In [ ]:
def generous_simplify(expr, tolerance=5e-3):
    expr = replace_near_integer(expr, tolerance=tolerance)
    expr = sympy.expand(expr, rational=False).evalf()
    expr = replace_near_integer(expr, tolerance=tolerance)
    expr = sympy.simplify(expr)
    expr = sympy.nsimplify(expr, tolerance=tolerance)
    return expr

In [ ]:
def count_by_threshold(df, threshold):
    return sum(df["mse"] < threshold)

In [ ]:
overall = {}

## Polynomials

Generally, the polynomial problems are easy.

### Lotka-Volterra (`d_lv1` and `d_lv2`)

In [ ]:
df_lv1 = get_data_set("d_lv1").sort_values("mse")
df_lv2 = get_data_set("d_lv2").sort_values("mse")

In [ ]:
df_lv1

In [ ]:
df_lv1["sympy_expr"].apply(generous_simplify)

In [ ]:
df_lv2["sympy_expr"].apply(generous_simplify)

After rounding, all of these are correct apart from debris.

In [ ]:
mse_threshold = 1e-11
ballpark_threshold = 5e-3

overall["lv1"] = {
    "data_set": "lv1",
    "perfect": 32,
    "mse_threshold": mse_threshold,
    "numerically_close": count_by_threshold(df_lv1, mse_threshold),
    "ballpark_threshold": ballpark_threshold,
    "ballpark": count_by_threshold(df_lv1, ballpark_threshold),
}
overall["lv2"] = {
    "data_set": "lv2",
    "perfect": 32,
    "mse_threshold": mse_threshold,
    "numerically_close": count_by_threshold(df_lv2, mse_threshold),
    "ballpark_threshold": ballpark_threshold,
    "ballpark": count_by_threshold(df_lv2, ballpark_threshold),
}

### Van der Pol (`d_vdp1` and `d_vdp2`)

In [ ]:
df_vdp1 = get_data_set("d_vdp1").sort_values("mse")
df_vdp2 = get_data_set("d_vdp2").sort_values("mse")

In [ ]:
df_vdp1["sympy_expr"].apply(generous_simplify)

In [ ]:
df_vdp2["sympy_expr"].apply(generous_simplify)

After rounding, all of these are correct.

In [ ]:
mse_threshold = 1e-11
ballpark_threshold = 5e-3

overall["vdp1"] = {
    "data_set": "vdp1",
    "perfect": 32,
    "mse_threshold": mse_threshold,
    "numerically_close": count_by_threshold(df_vdp1, mse_threshold),
    "ballpark_threshold": ballpark_threshold,
    "ballpark": count_by_threshold(df_vdp1, ballpark_threshold),
}
overall["vdp2"] = {
    "data_set": "vdp2",
    "perfect": 32,
    "mse_threshold": mse_threshold,
    "numerically_close": count_by_threshold(df_vdp2, mse_threshold),
    "ballpark_threshold": ballpark_threshold,
    "ballpark": count_by_threshold(df_vdp2, ballpark_threshold),
}

## Rational functions

### Predator-prey (`d_predprey1` and `d_predprey2`)

In [ ]:
df_predprey1 = get_data_set("d_predprey1").sort_values("mse")
df_predprey2 = get_data_set("d_predprey2").sort_values("mse")

In [ ]:
df_predprey1

In [ ]:
sns.histplot(df_predprey1["mse"], log_scale=True)

In [ ]:
expr = df_predprey1["sympy_expr"].iloc[0]
expr = replace_near_integer(expr, tolerance=5e-4)
expr = sympy.simplify(expr)
expr

In [ ]:
df_predprey1["sympy_expr"].apply(lambda e: generous_simplify(e, tolerance=5e-2))

There are 17 clearly good. Some of them take a little more tolerance.

In [ ]:
expr = df_predprey1["sympy_expr"].iloc[2]
expr = replace_near_integer(expr, tolerance=5e-2)
expr = sympy.simplify(expr)
expr

In [ ]:
expr = df_predprey1["sympy_expr"].iloc[14]
expr = 1 / sympy.simplify(1.0e-127 / sympy.simplify(expr * 1.0e-127))
# expr = replace_near_integer(expr, tolerance=5e-2)
# expr =sympy.simplify(expr)
expr = replace_near_integer(expr, tolerance=5e-2)
expr

In [ ]:
mse_threshold = 1e-7
ballpark_threshold = 5e-3
overall["predprey1"] = {
    "data_set": "predprey1",
    "perfect": 17,
    "mse_threshold": mse_threshold,
    "numerically_close": count_by_threshold(df_predprey1, mse_threshold),
    "ballpark_threshold": ballpark_threshold,
    "ballpark": count_by_threshold(df_predprey1, ballpark_threshold),
}

In [ ]:
df_predprey2

In [ ]:
sns.histplot(df_predprey2["mse"], log_scale=True)

In [ ]:
sympy.nsimplify(0.075)

In [ ]:
df_predprey2["sympy_expr"].apply(
    lambda e: replace_near_integer(e.evalf(), tolerance=5e-4).evalf()
)

In [ ]:
df_predprey2["sympy_expr"].apply(
    lambda e: sympy.cancel(generous_simplify(e, tolerance=5e-3))
)

In [ ]:
mse_threshold = 1e-7
ballpark_threshold = 5e-3
overall["predprey2"] = {
    "data_set": "predprey2",
    "perfect": 22,
    "mse_threshold": mse_threshold,
    "numerically_close": count_by_threshold(df_predprey2, mse_threshold),
    "ballpark_threshold": ballpark_threshold,
    "ballpark": count_by_threshold(df_predprey2, ballpark_threshold),
}

In [ ]:
overall

### Bacterial respiration (`d_bacres1` and `d_bacres2`)

In [ ]:
df_bacres1 = get_data_set("d_bacres1").sort_values("mse")
df_bacres2 = get_data_set("d_bacres2").sort_values("mse")

In [ ]:
df_bacres1

In [ ]:
sns.histplot(df_bacres1["mse"], log_scale=True)

In [ ]:
df_bacres1["sympy_expr"].iloc[0:5].apply(to_spiffy)

In [ ]:
replace_near_integer(df_bacres1["sympy_expr"].iloc[1], tolerance=1e-3)

In [ ]:
generous_simplify(df_bacres1["sympy_expr"].iloc[0])

In [ ]:
generous_simplify(df_bacres1["sympy_expr"].iloc[2])

In [ ]:
df_bacres1["sympy_expr"].iloc[0:5].apply(generous_simplify)

After rounding, the best one is perfectly correct.
Except for 2, all the others are numerically close.
All are rational functions or polynomials.

In [ ]:
mse_threshold = 1e-5
ballpark_threshold = 5e-3
overall["bacres1"] = {
    "data_set": "bacres1",
    "perfect": 1,
    "mse_threshold": mse_threshold,
    "numerically_close": count_by_threshold(df_bacres1, mse_threshold),
    "ballpark_threshold": ballpark_threshold,
    "ballpark": count_by_threshold(df_bacres1, ballpark_threshold),
}

In [ ]:
df_bacres2

In [ ]:
sns.histplot(df_bacres2["mse"], log_scale=True)

In [ ]:
expr = replace_near_integer(df_bacres2["sympy_expr"].iloc[0])
sympy.simplify(expr)

In [ ]:
expr = replace_near_integer(df_bacres2["sympy_expr"].iloc[1], tolerance=1e-4)
sympy.simplify(expr)

In [ ]:
df_bacres2["sympy_expr"].iloc[0:10].apply(
    lambda e: generous_simplify(e, tolerance=5e-2)
)

Two perfect, rest aren't bad, some have trig function junk.

In [ ]:
mse_threshold = 1e-5
ballpark_threshold = 5e-3
overall["bacres2"] = {
    "data_set": "bacres2",
    "perfect": 2,
    "mse_threshold": mse_threshold,
    "numerically_close": count_by_threshold(df_bacres2, mse_threshold),
    "ballpark_threshold": ballpark_threshold,
    "ballpark": count_by_threshold(df_bacres2, ballpark_threshold),
}

In [ ]:
overall

## Trig

### Bar magnets (`d_barmag1` and `d_barmag2`)

In [ ]:
df_barmag1 = get_data_set("d_barmag1").sort_values("mse")
df_barmag2 = get_data_set("d_barmag2").sort_values("mse")

In [ ]:
df_barmag1

In [ ]:
sns.histplot(df_barmag1["mse"], log_scale=True)

In [ ]:
df_barmag1["sympy_expr"].iloc[0:10].apply(
    lambda e: generous_simplify(e, tolerance=5e-2)
)

The first six are perfect.
The seventh has $\cos(x - \pi/2(1+\epsilon))$ instead of $\sin(x)$.
The eighth has $\cos(x - y +\pi/2(1+\epsilon))$ instead of $\sin(x - y)$, and it has something like a difference quotient of $\cos(x)$.

In [ ]:
df_barmag1["sympy_expr"].iloc[10:15].apply(
    lambda e: generous_simplify(e, tolerance=5e-2)
)

In [ ]:
mse_threshold = 1e-5
ballpark_threshold = 5e-3
overall["barmag1"] = {
    "data_set": "barmag1",
    "perfect": 7,
    "mse_threshold": mse_threshold,
    "numerically_close": count_by_threshold(df_barmag1, mse_threshold),
    "ballpark_threshold": ballpark_threshold,
    "ballpark": count_by_threshold(df_barmag1, ballpark_threshold),
}

In [ ]:
df_barmag2

In [ ]:
sns.histplot(df_barmag2["mse"], log_scale=True)

In [ ]:
df_barmag2["sympy_expr"].iloc[0:11].apply(
    lambda e: generous_simplify(e, tolerance=5e-2)
)

In [ ]:
df_barmag2["sympy_expr"].iloc[11:15].apply(
    lambda e: generous_simplify(e, tolerance=5e-2)
)

The ten out of the first eleven are perfect, or close enough.
Again, some difference quotients, and the remaining one has some junk.

In [ ]:
df_barmag1["sympy_expr"].iloc[10:15].apply(
    lambda e: generous_simplify(e, tolerance=5e-2)
)

In [ ]:
mse_threshold = 1e-5
ballpark_threshold = 5e-3
overall["barmag2"] = {
    "data_set": "barmag2",
    "perfect": 10,
    "mse_threshold": mse_threshold,
    "numerically_close": count_by_threshold(df_barmag2, mse_threshold),
    "ballpark_threshold": ballpark_threshold,
    "ballpark": count_by_threshold(df_barmag2, ballpark_threshold),
}

### Glider (`d_glider1` and `d_glider2`)

In [ ]:
df_glider1 = get_data_set("d_glider1").sort_values("mse")
df_glider2 = get_data_set("d_glider2").sort_values("mse")

In [ ]:
df_glider1

In [ ]:
sns.histplot(df_glider1["mse"], log_scale=True)

In [ ]:
df_glider1["sympy_expr"].iloc[0:10].apply(
    lambda e: generous_simplify(e, tolerance=5e-2)
)

The first two are perfect.

In [ ]:
mse_threshold = 1e-5
ballpark_threshold = 5e-3
overall["glider1"] = {
    "data_set": "glider1",
    "perfect": 2,
    "mse_threshold": mse_threshold,
    "numerically_close": count_by_threshold(df_glider1, mse_threshold),
    "ballpark_threshold": ballpark_threshold,
    "ballpark": count_by_threshold(df_glider1, ballpark_threshold),
}

In [ ]:
df_glider2

In [ ]:
sns.histplot(df_glider2["mse"], log_scale=True)

In [ ]:
df_glider2["sympy_expr"].iloc[0:10].apply(
    lambda e: generous_simplify(e, tolerance=5e-2)
)

Nope.

In [ ]:
mse_threshold = 1e-5
ballpark_threshold = 5e-3
overall["glider2"] = {
    "data_set": "glider2",
    "perfect": 0,
    "mse_threshold": mse_threshold,
    "numerically_close": count_by_threshold(df_glider2, mse_threshold),
    "ballpark_threshold": ballpark_threshold,
    "ballpark": count_by_threshold(df_glider2, ballpark_threshold),
}

### Shear flow (`d_shearflow1` and `d_shearflow2`)

In [ ]:
df_shearflow1 = get_data_set("d_shearflow1").sort_values("mse")
df_shearflow2 = get_data_set("d_shearflow2").sort_values("mse")

In [ ]:
df_shearflow1

In [ ]:
sns.histplot(df_shearflow1["mse"], log_scale=True)

In [ ]:
df_shearflow1["sympy_expr"].iloc[
    0:10
]  # .apply(lambda e: generous_simplify(e, tolerance=5e-2))

Nope.

In [ ]:
mse_threshold = 1e-5
ballpark_threshold = 5e-3
overall["shearflow1"] = {
    "data_set": "shearflow1",
    "perfect": 0,
    "mse_threshold": mse_threshold,
    "numerically_close": count_by_threshold(df_shearflow1, mse_threshold),
    "ballpark_threshold": ballpark_threshold,
    "ballpark": count_by_threshold(df_shearflow1, ballpark_threshold),
}

In [ ]:
df_shearflow2

In [ ]:
sns.histplot(df_shearflow2["mse"], log_scale=True)

In [ ]:
df_shearflow2["sympy_expr"].iloc[0:3].apply(
    lambda e: generous_simplify(e, tolerance=5e-2)
)

The first one isn't totally awful.

In [ ]:
mse_threshold = 1e-5
ballpark_threshold = 5e-3
overall["shearflow2"] = {
    "data_set": "shearflow2",
    "perfect": 0,
    "mse_threshold": mse_threshold,
    "numerically_close": count_by_threshold(df_shearflow2, mse_threshold),
    "ballpark_threshold": ballpark_threshold,
    "ballpark": count_by_threshold(df_shearflow2, ballpark_threshold),
}

### Shear flow grid (`d_sfgrid1` and `d_sfgrid2`)

In [ ]:
df_sfgrid1 = get_data_set("d_sfgrid1").sort_values("mse")
df_sfgrid2 = get_data_set("d_sfgrid2").sort_values("mse")

In [ ]:
df_sfgrid1

In [ ]:
sns.histplot(df_sfgrid1["mse"], log_scale=True)

In [ ]:
df_sfgrid1["sympy_expr"].iloc[
    0:10
]  # .apply(lambda e: generous_simplify(e, tolerance=5e-2))

Nope.

In [ ]:
mse_threshold = 1e-5
ballpark_threshold = 5e-3
overall["sfgrid1"] = {
    "data_set": "sfgrid1",
    "perfect": 0,
    "mse_threshold": mse_threshold,
    "numerically_close": count_by_threshold(df_sfgrid1, mse_threshold),
    "ballpark_threshold": ballpark_threshold,
    "ballpark": count_by_threshold(df_sfgrid1, ballpark_threshold),
}

In [ ]:
df_sfgrid2

In [ ]:
sns.histplot(df_sfgrid2["mse"], log_scale=True)

In [ ]:
df_sfgrid2["sympy_expr"].iloc[0:3].apply(lambda e: generous_simplify(e, tolerance=5e-2))

 The first one is actually correct.

In [ ]:
mse_threshold = 1e-5
ballpark_threshold = 5e-3
overall["sfgrid2"] = {
    "data_set": "sfgrid2",
    "perfect": 1,
    "mse_threshold": mse_threshold,
    "numerically_close": count_by_threshold(df_sfgrid2, mse_threshold),
    "ballpark_threshold": ballpark_threshold,
    "ballpark": count_by_threshold(df_sfgrid2, ballpark_threshold),
}

## Overall

In [ ]:
pd.DataFrame(overall).T